# G17 — BrEaST diagnosis: is the drop caused by brightness/contrast? + test-time intensity fixes
Inputs: **G17_loss_search_R4** output + **Breast-Lesions-USG**. CPU-friendly: one model (R4_upp_dice_s42, best single on BrEaST), plain prediction, Dice/IoU/fail only.

In [ ]:
!pip -q install segmentation-models-pytorch safetensors datasets

In [ ]:
import glob, os, re, shutil, sys, numpy as np, pandas as pd, cv2, torch
torch.set_num_threads(os.cpu_count())
src = sorted(glob.glob('/kaggle/input/**/g17core.py', recursive=True))[0]; shutil.copy(src, 'g17core.py')
sys.path.insert(0, '.'); import g17core as g
RUNS = sorted(d for d in glob.glob('/kaggle/input/**/runs', recursive=True) if os.path.isdir(d))[0]
pngs = [p for p in glob.glob('/kaggle/input/**/*.png', recursive=True) if '/runs/' not in p]
imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in pngs if not re.search(r'_(tumou?r|other|mask)', os.path.basename(p), re.I)}
groups = {}
for m in [p for p in pngs if re.search(r'_tumou?r', os.path.basename(p), re.I)]:
    groups.setdefault(re.split(r'_tumou?r', os.path.splitext(os.path.basename(m))[0], flags=re.I)[0], []).append(m)
ids, RAW, MSK = [], [], []
for k in sorted(k for k in groups if k in imgs):
    im = cv2.imread(imgs[k], cv2.IMREAD_GRAYSCALE); mk = np.zeros(im.shape, np.uint8)
    for m in groups[k]:
        mm = cv2.imread(m, cv2.IMREAD_GRAYSCALE)
        if mm.shape != im.shape: mm = cv2.resize(mm, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
        mk |= ((mm > 127) * 255).astype(np.uint8)
    if mk.max() == 0: continue
    ids.append(k); RAW.append(cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA)); MSK.append(cv2.resize(mk, (256, 256), interpolation=cv2.INTER_NEAREST))
RAW, MSK = np.stack(RAW), np.stack(MSK)
print('BrEaST', len(ids))
# training-set intensity reference (same 256x256 raw images the models saw)
split = g.load_splits(); tr = g.to_tensors(split['train'], 'raw', 'train')['img'][:, 0].cpu().numpy()
ref_hist = np.bincount(tr.ravel(), minlength=256).astype(np.float64); ref_cdf = np.cumsum(ref_hist) / ref_hist.sum()
def stats(a):  # per-image stats inside the non-black (scan) area
    v = a[a > 8] if (a > 8).sum() > 500 else a.ravel(); return v.mean(), v.std(), np.percentile(v, 50)
tr_stats = np.array([stats(a) for a in tr[:800]]); ex_stats = np.array([stats(a) for a in RAW])
print('mean / std / median brightness  TRAIN %.1f %.1f %.1f | BrEaST %.1f %.1f %.1f' % (*tr_stats.mean(0), *ex_stats.mean(0)))

In [ ]:
def hist_match(a):
    h = np.bincount(a.ravel(), minlength=256).astype(np.float64); cdf = np.cumsum(h) / h.sum()
    lut = np.interp(cdf, ref_cdf, np.arange(256)).astype(np.uint8); return lut[a]
def clahe(a): return cv2.createCLAHE(2.0, (8, 8)).apply(a)
def gamma_to_train(a):
    m = stats(a)[0] / 255; t = tr_stats[:, 0].mean() / 255
    gma = np.log(t) / np.log(max(m, 1e-3)); return (255 * (a / 255.0) ** gma).clip(0, 255).astype(np.uint8)
def pstretch(a):
    lo, hi = np.percentile(a[a > 8] if (a > 8).sum() > 500 else a, (1, 99)); return ((a.astype(np.float32) - lo) / max(hi - lo, 1) * 255).clip(0, 255).astype(np.uint8)
TF = {'none': lambda a: a, 'hist-match to train': hist_match, 'gamma to train mean': gamma_to_train, 'percentile stretch 1-99': pstretch, 'CLAHE': clahe}
model = g.build_model({"arch": "smp:UnetPlusPlus:resnet34"}).to(g.DEVICE)
model.load_state_dict(torch.load(os.path.join(RUNS, 'R4_upp_dice_s42', 'best.pt'), map_location=g.DEVICE)); model.eval()
gt = MSK > 127; rows, per = [], {}
for name, fn in TF.items():
    X = np.stack([fn(a) for a in RAW])
    data = {"img": torch.from_numpy(X)[:, None].to(g.DEVICE), "msk": torch.from_numpy(MSK)[:, None].to(g.DEVICE), "dataset": ['BrEaST'] * len(ids), "image_id": ids}
    with torch.no_grad(): p = g.predict(model, data) >= 0.5
    tp = (p & gt).sum((1, 2)); fp = (p & ~gt).sum((1, 2)); fn_ = (~p & gt).sum((1, 2))
    d = (2 * tp + 1e-7) / (2 * tp + fp + fn_ + 1e-7); per[name] = d
    rows.append(dict(transform=name, dice=100 * d.mean(), iou=100 * ((tp + 1e-7) / (tp + fp + fn_ + 1e-7)).mean(), fail_lt50=100 * (d < 0.5).mean(),
                     over_seg=100 * (p.sum((1, 2)) > 1.5 * gt.sum((1, 2))).mean(), missed=100 * (p.sum((1, 2)) == 0).mean()))
    print(rows[-1])
print('=== BrEaST TEST-TIME INTENSITY FIXES (R4_upp_dice_s42, plain) ===')
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
# Is failure linked to brightness / contrast? (Spearman) + Dice by brightness tercile
from scipy.stats import spearmanr
d0 = per['none']; m, s = ex_stats[:, 0], ex_stats[:, 1]
print('Spearman Dice vs brightness: rho=%.2f p=%.3g' % spearmanr(d0, m))
print('Spearman Dice vs contrast(std): rho=%.2f p=%.3g' % spearmanr(d0, s))
q = pd.qcut(m, 3, labels=['darker', 'middle', 'brighter'])
print(pd.DataFrame({'tercile': q, 'dice': d0 * 100}).groupby('tercile').dice.agg(['size', 'mean']).round(2))
pd.DataFrame({'image_id': ids, 'brightness': m, 'contrast': s, **{k: v for k, v in per.items()}}).to_csv('breast_intensity_diagnosis.csv', index=False)